# Module 10 — Sensitivity, Placebo & Falsification Tests

Part of the Causal Inference Playbook. Course program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

- What this module does: it builds no new estimate. It takes an estimate an earlier method already made and checks whether to trust it. The earlier modules each built an estimate of an effect. This module checks one, and the same checks apply to an estimate from any of those methods.
- The question: without an experiment, the right answer is never known. So how can an analyst tell that an estimate of an effect is wrong?
- The example: a job-training program, the NSW (National Supported Work) program, whose right answer is known from a lottery. The method under test is a regression that compares the men who took the program with men who didn't, holding fixed the men's age, schooling, earnings before the program and other columns in the data.
- Bias: the method's estimate minus the program's true effect. A method with no bias gets the true effect, apart from chance.
- The way: hide the right answer and run three checks that need no right answer. Then compare what the checks said with the right answer.
- Two kinds of steps, each with a tag at its top:
  - [in real life]: a step an analyst without a lottery runs.
  - [only in this module]: a step that uses the lottery men or the right answer, which a real analyst doesn't have, to grade the checks.
- The three checks:
  1. A placebo outcome: estimate the program's effect on something the program couldn't change, such as earnings in a year before the program. The true effect there is 0, so an estimate whose 95% confidence interval leaves out 0 is a warning of bias. Such a check is also called a falsification test: a test that can show the method is wrong.
  2. A stand-in group: a group that didn't take the program but stands in for the men who took it. Estimate the "effect" between the stand-in group and the comparison group, which both didn't take the program. The true effect is 0, so the estimate shows what the method reports when the program does nothing: an estimate of the method's bias.
  3. Sensitivity to a missing column: leave out each column the regression uses, one at a time, and measure how far the estimate moves. The columns in the data are the only measured examples of how far one column can move the estimate, so they give a size to compare with, not a limit, for how far a column the data doesn't have might move it.

## From the curriculum

> ## <u>Module 10 — Sensitivity, Placebo & Falsification Tests</u>
>
> **Concepts**
> - Placebo outcome (a falsification test): an outcome the program couldn't
>   have changed, such as earnings in a year before the program. The method's
>   estimate of the effect on that outcome should be 0. An estimate whose 95%
>   confidence interval leaves out 0 shows that the comparison group differs
>   from the men who took the program in ways the columns don't capture.
> - A stand-in group: a group that didn't get the program but stands in for the
>   men who took it, such as applicants a lottery kept out. The method's
>   estimate of the "effect" between the stand-in group and the comparison group
>   should be 0, and any estimate is an estimate of the method's bias.
> - Move from leaving out a column: how far an estimate moves when one column is left
>   out of the regression. Move = (the column's coefficient in the full
>   regression) × (the column difference: the difference in that column between
>   the men who took the program and the comparison group, holding the other columns
>   fixed).
> - Sensitivity to a missing column: the columns in the data show how far one
>   column can move the estimate. A column the data doesn't have could move the
>   estimate as far. An estimate that one such column could erase is fragile.
>
> **Why it matters**
> Outside an experiment, the right answer is never known, so an estimate can't
> be checked against it. These checks need no right answer: they ask whether the
> method finds effects where none can exist, and how far a missing column could
> move the estimate. No check proves an estimate right. A failed check shows a
> reason to doubt it, and a passed check shows only that the check couldn't find
> a problem.
>
> **Worked example**
> The NSW (National Supported Work) job-training program, with the PSID (Panel
> Study of Income Dynamics) men as the comparison group, where the right answer
> is known from the program's lottery. A regression estimate of the program's
> effect on 1978 earnings, then three checks with the right answer hidden: a
> placebo outcome (1975 earnings), a stand-in group (the lottery men against
> the PSID men), and the move from leaving out each column. Finally, the
> checks against the right answer.
>
> **Resources**
> - [The Effect, ch. 8 "Causal Paths and Closing Back Doors"](https://theeffectbook.net/ch-CausalPaths.html)
>   (Nick Huntington-Klein, ~20 min for section 8.6 "Using Paths to Test Your
>   Diagram", verified): placebo tests, which epidemiology calls falsification
>   tests.
> - [The Effect, ch. 13 "Regression"](https://theeffectbook.net/ch-StatisticalAdjustment.html)
>   (Nick Huntington-Klein, ~15 min for section 13.1.2 "Regression Assumptions
>   and Sampling Variation", verified): omitted variable bias, the move from
>   leaving out a column.
> - [Cinelli and Hazlett (2020), "Making Sense of Sensitivity: Extending Omitted Variable Bias"](https://doi.org/10.1111/rssb.12348)
>   (Journal of the Royal Statistical Society, Series B, verified): the standard
>   modern method for sensitivity to a missing column, including comparing a
>   missing column with the columns in the data.
>
> **Exercise type:** the same three checks with the CPS (Current Population
> Survey) men as the comparison group, then the checks against the right answer.
> Fully worked.
>
> **Interview angle:** "Matching says users who turned on feature X spend 8%
> more. An experiment isn't possible. How would you decide whether to trust the
> 8%?"

## The data

The NSW job-training program, from the file by Rajeev Dehejia and Sadek Wahba. Each row is one man. Three groups of men:
- The treated men: 185 men who applied for NSW, were let in by a lottery, and took the program in 1976 or 1977.
- The lottery men: 260 men who applied for NSW like the treated men, but whom the lottery kept out. The lottery, not the men, decided who got in, so the treated men and the lottery men differ only by chance, and the lottery men show what the treated men would have earned without the program.
- The PSID men: 2,490 men from the PSID (Panel Study of Income Dynamics), a US survey. None of them took the program.

The columns: age, years of education, four yes/no columns (1 = yes, 0 = no): black, hispanic, married, no high-school degree, earnings in 1974 and in 1975 (both before the program), and earnings in 1978 (after the program). Earnings are in US dollars per year.

The exercise uses a fourth group, the CPS men: 15,992 men from the CPS (Current Population Survey), another US survey, none of whom took the program.

The output counts each group and checks that no row has a missing value, because a regression drops such rows without saying so, and every man should count.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

import common

experiment = common.load_lalonde("experimental")  # 185 treated men (treat = 1) and 260 lottery men (treat = 0)
psid = common.load_lalonde("psid")                # 185 treated men (treat = 1) and 2,490 PSID men (treat = 0)
cps = common.load_lalonde("cps")                  # 185 treated men (treat = 1) and 15,992 CPS men (treat = 0)
treated, lottery = experiment[experiment.treat == 1], experiment[experiment.treat == 0]
COLUMNS = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]
SEVEN = [column for column in COLUMNS if column != "re75"]
NAMES = {"age": "age", "education": "years of education", "black": "black", "hispanic": "hispanic", "married": "married",
         "nodegree": "no high-school degree", "re74": "1974 earnings", "re75": "1975 earnings", "re78": "1978 earnings"}

print(f"Treated men = {len(treated)}")
print(f"Lottery men = {len(lottery)}")
print(f"PSID men = {(psid.treat == 0).sum():,}")
print(f"CPS men = {(cps.treat == 0).sum():,}")
print(f"Rows with a missing value = {sum(df.isna().any(axis=1).sum() for df in (experiment, psid, cps))}")

Treated men = 185
Lottery men = 260
PSID men = 2,490
CPS men = 15,992
Rows with a missing value = 0


## What estimate do the checks test?

**[only in this module]** First the right answer, from the lottery: the treated men's mean 1978 earnings minus the lottery men's mean 1978 earnings. The two groups differ only by chance, so the difference estimates the program's effect on the treated men with no bias, and with only chance left in it.

In [2]:
right_answer = round(treated.re78.mean(), 1) - round(lottery.re78.mean(), 1)
print("Right answer =")
print(f"  = (mean 1978 earnings, {len(treated)} treated men) - (mean 1978 earnings, {len(lottery)} lottery men) =")
print(f"  = {treated.re78.mean():,.1f} - {lottery.re78.mean():,.1f} = {right_answer:,.1f} dollars per year")

Right answer =
  = (mean 1978 earnings, 185 treated men) - (mean 1978 earnings, 260 lottery men) =
  = 6,349.1 - 4,554.8 = 1,794.3 dollars per year


**[in real life]** The estimate to test is a regression estimate, built the way an analyst without a lottery would build it: the analyst has the treated men and the PSID men only.
- A "took the program" column: 1 for each treated man, 0 for each man in the comparison group.
- One regression of 1978 earnings on "took the program" and the 8 columns from before the program: 1978 earnings = a + (b × took the program) + (c1 × age) + (c2 × years of education) + … + (c8 × 1975 earnings).
- The fitting function, a standard least-squares solver, returns b, the coefficient of "took the program": the difference in 1978 earnings between treated men and comparison men with the same values in all 8 columns. b is the regression estimate of the program's effect.
- The fitting function also returns b's standard error. The 95% confidence interval = b ± (1.96 × standard error), where 1.96 is the usual multiplier for 95%.

The output runs the regression twice:
1. **[only in this module]** With the lottery men as the comparison group. The analyst doesn't have the lottery men. This module has them only to know the right answer and grade the checks at the end. The lottery men show what the treated men would have earned without the program, so the regression with them should land near the right answer, with the right answer inside the lottery regression's interval.
2. **[in real life]** With the PSID men: the analyst's own estimate, the one the three checks test.

Each regression runs on the treated men and the comparison group together:
- "Took the program" = 1 for the 185 treated men, and 0 for each man in the comparison group: the 260 lottery men, or the 2,490 PSID men.
- b is the gap in predicted 1978 earnings between those two groups, and a gap needs both groups. So:
  - The lottery regression runs on 185 + 260 = 445 men.
  - The PSID regression runs on 185 + 2,490 = 2,675 men.

In [3]:
def estimate(df, outcome, columns, title, column="took the program", quiet=False):
    # Returns the coefficient of `treat` and its 95% interval, each rounded to one decimal.
    model = smf.ols(f"{outcome} ~ treat + " + " + ".join(columns), df).fit()
    value, error = round(model.params["treat"], 1), round(model.bse["treat"], 1)
    half = round(1.96 * error, 1)
    low, high = round(value - half, 1), round(value + half, 1)
    if not quiet:
        print(f"{title} ({len(df):,} men) =")
        print(f"  = coefficient of '{column}' (one regression of {NAMES[outcome]} on '{column}' and {len(columns)} columns) =")
        print(f"  = {value:,.1f} dollars per year")
        print(f"The result, {value:,.1f}, is b in: a + (b x {column}) + (c1 x {NAMES[columns[0]]}) + ... + (c{len(columns)} x {NAMES[columns[-1]]})")
        print()
        print(f"Standard error, returned by the fitting function = {error:,.1f}")
        print()
        print("95% confidence interval =")
        print("  = (estimate) ± (1.96 x standard error) =")
        print(f"  = {value:,.1f} ± (1.96 x {error:,.1f}) = {value:,.1f} ± {half:,.1f} =")
        print(f"  = {low:,.1f} to {high:,.1f}")
    return value, low, high

lottery_estimate = estimate(experiment, "re78", COLUMNS, "Lottery men as the comparison group, regression estimate")

Lottery men as the comparison group, regression estimate (445 men) =
  = coefficient of 'took the program' (one regression of 1978 earnings on 'took the program' and 8 columns) =
  = 1,676.3 dollars per year
The result, 1,676.3, is b in: a + (b x took the program) + (c1 x age) + ... + (c8 x 1975 earnings)

Standard error, returned by the fitting function = 638.7

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = 1,676.3 ± (1.96 x 638.7) = 1,676.3 ± 1,251.9 =
  = 424.4 to 2,928.2


In [4]:
psid_estimate = estimate(psid, "re78", COLUMNS, "PSID men as the comparison group, regression estimate")

PSID men as the comparison group, regression estimate (2,675 men) =
  = coefficient of 'took the program' (one regression of 1978 earnings on 'took the program' and 8 columns) =
  = 751.9 dollars per year
The result, 751.9, is b in: a + (b x took the program) + (c1 x age) + ... + (c8 x 1975 earnings)

Standard error, returned by the fitting function = 915.3

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = 751.9 ± (1.96 x 915.3) = 751.9 ± 1,794.0 =
  = -1,042.1 to 2,545.9


**[in real life]** The 10 numbers of the PSID regression, b among them:

In [5]:
full_psid = smf.ols("re78 ~ treat + " + " + ".join(COLUMNS), psid).fit()
rows = [("a", "(none, added to every man)", full_psid.params["Intercept"]),
        ("b", "took the program", full_psid.params["treat"])]
rows += [(f"c{i}", NAMES[column], full_psid.params[column]) for i, column in enumerate(COLUMNS, start=1)]
print(f"PSID men as the comparison group, the 10 numbers the fitting function returns ({len(psid):,} men)")
print()
print(f"{'Number':<8} {'Multiplies':<28} {'Value':>14}")
for number, multiplies, value in rows:
    print(f"{number:<8} {multiplies:<28} {value:>14,.6f}")

PSID men as the comparison group, the 10 numbers the fitting function returns (2,675 men)

Number   Multiplies                            Value
a        (none, added to every man)      -129.742670
b        took the program                 751.946356
c1       age                              -83.565585
c2       years of education               592.610194
c3       black                           -570.927981
c4       hispanic                       2,163.281159
c5       married                        1,240.519511
c6       no high-school degree            590.466939
c7       1974 earnings                      0.278121
c8       1975 earnings                      0.568091


**[only in this module]** The miss = how far the PSID estimate is from the right answer: the size of the error the checks should find. A real analyst can't compute the miss, because it needs the right answer.

In [6]:
miss = round(psid_estimate[0] - right_answer, 1)
print("PSID men, miss =")
print("  = (regression estimate, PSID men) - (right answer) =")
print(f"  = {psid_estimate[0]:,.1f} - {right_answer:,.1f} = {miss:,.1f} dollars per year")

PSID men, miss =
  = (regression estimate, PSID men) - (right answer) =
  = 751.9 - 1,794.3 = -1,042.4 dollars per year


**[only in this module]** **Observation:** with the lottery men, the regression estimate is $1,676.3, and the right answer, $1,794.3, lies inside the interval, 424.4 to 2,928.2, as we want. The two numbers are two estimates of the same effect: the right answer is a plain difference of means, while the regression also holds the 8 columns fixed. With the PSID men, the regression estimate is $751.9, a miss of (−$1,042.4). The PSID estimate's interval, (−1,042.1) to 2,545.9, holds both 0 and the right answer, so the PSID estimate alone can't say whether the program changed earnings at all. Two pairs of numbers here are close by coincidence and unrelated: the interval's lower end, (−1,042.1), and the miss, (−1,042.4); and the interval's half-width, 1,794.0, and the right answer, 1,794.3.

The right answer is itself measured on 445 men, so it carries some chance too, and part of the miss can be chance. This module still uses $1,794.3 as the right answer, because it is the only estimate of the effect with no bias from the comparison group.

From here on, the right answer is hidden: in real work, an analyst has the treated men and a survey such as the PSID, and nothing tells the analyst that $751.9 misses by (−$1,042.4). Each check runs on the PSID men, the group under test, and, where the check allows it, on the lottery men as the comparison group too, to show what the check gives for a comparison group whose estimate lands near the right answer.

## Check 1 of 3: does the method find an effect on 1975 earnings, which the program couldn't change?

The treated men took the program in 1976 or 1977, so the program can't have changed their 1975 earnings: the program's true effect on 1975 earnings is 0. A placebo outcome uses exactly such an outcome.

The check runs the same regression with 1975 earnings on the left side, in place of 1978 earnings. The right side keeps "took the program" and the other 7 columns, without 1975 earnings: with 1975 earnings on both sides, 1975 earnings would predict itself exactly, and the coefficient of "took the program" would be 0 whatever the data. The coefficient of "took the program" is then the difference in 1975 earnings between treated men and comparison men with the same values in the other 7 columns.

The output runs check 1 twice:
1. **[only in this module]** The first output uses the lottery men.
2. **[in real life]** The second output uses the PSID men.

What we want: the 95% interval includes 0, because the check looks for no difference. This is the standard rule. An interval without 0 is a warning: the 7 columns don't make the comparison men like the treated men in 1975 earnings, which raises the doubt that the 8 columns make the comparison men like the treated men in what the treated men would have earned in 1978 without the program.

In [7]:
lottery_placebo = estimate(experiment, "re75", SEVEN, "Lottery men as the comparison group, placebo estimate (effect on 1975 earnings)")

Lottery men as the comparison group, placebo estimate (effect on 1975 earnings) (445 men) =
  = coefficient of 'took the program' (one regression of 1975 earnings on 'took the program' and 7 columns) =
  = 317.9 dollars per year
The result, 317.9, is b in: a + (b x took the program) + (c1 x age) + ... + (c7 x 1974 earnings)

Standard error, returned by the fitting function = 225.3

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = 317.9 ± (1.96 x 225.3) = 317.9 ± 441.6 =
  = -123.7 to 759.5


In [8]:
psid_placebo = estimate(psid, "re75", SEVEN, "PSID men as the comparison group, placebo estimate (effect on 1975 earnings)")

PSID men as the comparison group, placebo estimate (effect on 1975 earnings) (2,675 men) =
  = coefficient of 'took the program' (one regression of 1975 earnings on 'took the program' and 7 columns) =
  = -1,781.6 dollars per year
The result, -1,781.6, is b in: a + (b x took the program) + (c1 x age) + ... + (c7 x 1974 earnings)

Standard error, returned by the fitting function = 642.3

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = -1,781.6 ± (1.96 x 642.3) = -1,781.6 ± 1,258.9 =
  = -3,040.5 to -522.7


**Observation:** with the lottery men, the placebo estimate is $317.9, and the interval, (−123.7) to 759.5, includes 0, so the check doesn't warn. With the PSID men, the placebo estimate is (−$1,781.6), and the interval, (−3,040.5) to (−522.7), doesn't include 0, so the check warns: the PSID regression estimate may have a bias. The treated men earned $1,781.6 less in 1975 than PSID men of the same age, education, race, marriage, degree and 1974 earnings.

The regression of 1978 earnings holds 1975 earnings fixed, so the $1,781.6 difference itself doesn't enter the regression estimate. But whatever made the treated men earn less in 1975 than PSID men alike in the other 7 columns isn't in any column, and it can make the treated men earn less in 1978 too, apart from the program. The warning doesn't say how far the regression estimate is off, or in which direction.

**Conclusion:** check 1 warns with the PSID men, a bad sign: the columns don't make the PSID men like the treated men, so the PSID regression estimate is in doubt.

## Check 2 of 3: does the method find an effect between the lottery men and the PSID men, who both didn't take the program?

**[in real life]** A real analyst can run check 2 only with a stand-in group of their own, such as applicants who were turned away.

**[only in this module]** The lottery men are the stand-in group here. The module's analyst has only the treated men and the PSID men, so this module borrows the lottery men to play the stand-in group.

Check 2 needs a stand-in group: men who didn't take the program but who are like the treated men. Here, the lottery men are the stand-in group: they applied like the treated men, and only the lottery kept them out. In real work, such a group is rare: applicants turned away, or a waiting list. Check 2 uses the lottery men only as a group without the program, not to compute the right answer.

The check runs the same regression of 1978 earnings on the 8 columns, with the 260 lottery men in place of the 185 treated men, against the 2,490 PSID men:
- A "lottery man" column: 1 for each lottery man, 0 for each PSID man.
- Neither group took the program, so the true effect is 0.
- The lottery men stand in for the treated men without the program. So the coefficient of "lottery man" is what the regression with the PSID men reports when the program has no effect: the bias estimate.

What we want: the 95% interval includes 0, because the check looks for no difference. This is the standard rule. An interval without 0 is a warning. When the lottery men are the comparison group, check 2 can't run: the lottery men would be compared with themselves.

In [9]:
second = pd.concat([lottery.assign(treat=1), psid[psid.treat == 0]], ignore_index=True)
psid_bias = estimate(second, "re78", COLUMNS, "Lottery men against PSID men, bias estimate", column="lottery man")

Lottery men against PSID men, bias estimate (2,750 men) =
  = coefficient of 'lottery man' (one regression of 1978 earnings on 'lottery man' and 8 columns) =
  = -1,157.8 dollars per year
The result, -1,157.8, is b in: a + (b x lottery man) + (c1 x age) + ... + (c8 x 1975 earnings)

Standard error, returned by the fitting function = 832.4

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = -1,157.8 ± (1.96 x 832.4) = -1,157.8 ± 1,631.5 =
  = -2,789.3 to 473.7


**Observation:** the bias estimate is (−$1,157.8): the regression reports the lottery men as earning $1,157.8 less in 1978 than PSID men with the same 8 columns, with no program in either group. The interval, (−2,789.3) to 473.7, includes 0, so the check doesn't warn.

Not warning doesn't show that the bias is small. Every bias from (−$2,789.3) to $473.7 lies inside the interval, so the check can't tell a bias of 0 from a bias of (−$2,000). The check warns only when the bias is large enough, against the interval's width, that the whole interval lies on one side of 0.

**Conclusion:** with the PSID men, the stand-in group doesn't warn, but its interval is too wide to show that the bias is small.

## Check 3 of 3: how far could a missing column move the estimate?

**[in real life]** The regression holds 8 columns fixed. A column the data doesn't have, such as how keen each man was to find work, could differ between the treated men and the PSID men and affect 1978 earnings. Such a column can't be measured. But each column in the data can be left out, one at a time, to measure how far one column moves the estimate. A missing column is "as strong as" a column in the data when the missing column's (coefficient × column difference), defined in step 2 below, is as large as that column's. Nothing says that a missing column is weaker or stronger than the columns in the data: the columns in the data are only the measured examples.

The check has 3 steps:
- Step 1 of 3 leaves out one column, 1975 earnings.
- Step 2 of 3 shows where the move comes from.
- Step 3 of 3 repeats step 1 for all 8 columns.

### Step 1 of 3: the estimate without 1975 earnings

The same regression of 1978 earnings on "took the program" and 7 columns, without 1975 earnings, with the PSID men:

In [10]:
without_1975 = estimate(psid, "re78", SEVEN, "", quiet=True)[0]
print("PSID men as the comparison group, regression estimate without 1975 earnings (2,675 men) =")
print("  = coefficient of 'took the program' (one regression of 1978 earnings on 'took the program' and 7 columns) =")
print(f"  = {without_1975:,.1f} dollars per year")
print(f"The result, {without_1975:,.1f}, is b in: a + (b x took the program) + (c1 x age) + ... + (c7 x 1974 earnings)")

PSID men as the comparison group, regression estimate without 1975 earnings (2,675 men) =
  = coefficient of 'took the program' (one regression of 1978 earnings on 'took the program' and 7 columns) =
  = -260.2 dollars per year
The result, -260.2, is b in: a + (b x took the program) + (c1 x age) + ... + (c7 x 1974 earnings)


In [11]:
print("Move from leaving out 1975 earnings =")
print("  = (estimate without 1975 earnings) - (estimate with all 8 columns) =")
print(f"  = {without_1975:,.1f} - {psid_estimate[0]:,.1f} = {without_1975 - psid_estimate[0]:,.1f} dollars per year")

Move from leaving out 1975 earnings =
  = (estimate without 1975 earnings) - (estimate with all 8 columns) =
  = -260.2 - 751.9 = -1,012.1 dollars per year


### Step 2 of 3: where the move comes from

**[in real life]** Move from leaving out a column = (the column's coefficient in the 8-column regression) × (the column difference: the difference in the column between treated men and comparison men, holding the other 7 columns fixed).

For 1975 earnings:
- Coefficient of 1975 earnings: how many dollars of 1978 earnings come with each extra dollar earned in 1975, for men alike in everything else. The output prints it.
- Column difference in 1975 earnings: how much less the treated men earned in 1975 than PSID men alike in the other 7 columns. This is check 1's result, (−$1,781.6), used at 4 decimals: (−1,781.5808).

In [12]:
full = smf.ols("re78 ~ treat + " + " + ".join(COLUMNS), psid).fit()
coefficient_1975 = full.params["re75"]
difference_1975 = smf.ols("re75 ~ treat + " + " + ".join(SEVEN), psid).fit().params["treat"]
print("Coefficient of 1975 earnings in the 8-column regression, PSID men, returned by the fitting function =")
print(f"  = {coefficient_1975:.6f} dollars of 1978 earnings per dollar of 1975 earnings")
print()
print("Treated men - PSID men, column difference in 1975 earnings, other 7 columns fixed =")
print(f"  = {difference_1975:,.4f} dollars")
print(f"The result, {difference_1975:,.4f}, is check 1's placebo estimate.")
print()
print("Move from leaving out 1975 earnings =")
print("  = (coefficient of 1975 earnings) x (column difference in 1975 earnings) =")
print(f"  = {coefficient_1975:.6f} x ({difference_1975:,.4f}) = {coefficient_1975 * difference_1975:,.1f} dollars per year")

Coefficient of 1975 earnings in the 8-column regression, PSID men, returned by the fitting function =
  = 0.568091 dollars of 1978 earnings per dollar of 1975 earnings

Treated men - PSID men, column difference in 1975 earnings, other 7 columns fixed =
  = -1,781.5808 dollars
The result, -1,781.5808, is check 1's placebo estimate.

Move from leaving out 1975 earnings =
  = (coefficient of 1975 earnings) x (column difference in 1975 earnings) =
  = 0.568091 x (-1,781.5808) = -1,012.1 dollars per year


Why the move is the product. Check 1 measured the (−$1,781.6) among men alike in the other 7 columns, so the other 7 columns can't account for that part of the treated men's lower 1975 earnings: among men alike in the 7 columns, only "took the program" differs. In the 8-column regression, that lower 1975 earnings comes with 0.568091 × (−1,781.6) = (−$1,012.1) of 1978 earnings, through the coefficient of 1975 earnings. Leave 1975 earnings out, and the treated men's 1978 earnings are still $1,012.1 lower because of their lower 1975 earnings, but among men alike in the 7 columns, the only column left that differs between treated men and PSID men is "took the program". So the coefficient of "took the program" falls by $1,012.1. The appendix proves that the formula holds exactly.

### Step 3 of 3: every column, against the size of the estimate

The table repeats steps 1 and 2 for each of the 8 columns: the estimate without the column, the move, the column's coefficient, the column difference, and the product, which matches the move up to 0.1: the move subtracts two estimates that are each rounded to one decimal first. Coefficients are printed to 6 decimals, so each product checks by hand.

Each column difference is in its column's unit: years for age and education, a share for the yes/no columns, dollars for earnings. Each coefficient is in dollars of 1978 earnings per one unit of its column. For married, a column difference of (−0.5215) means that, comparing treated men and PSID men alike in the other 7 columns, the share married among the treated men is 52.15 percentage points below the share married among the PSID men.

A missing column could move the estimate up or down, so the sizes count, ignoring the sign. What we want: the largest move, ignoring the sign, smaller than the estimate itself. There is no standard cutoff. The standard modern method, by Cinelli and Hazlett, listed in the curriculum above, builds on the same formula with its own measures. The rule: an estimate of the program's effect is fragile when leaving out one column moves the estimate by more than its own size, because a missing column that moved the estimate as far could push it below 0 and flip the answer.

The output prints 2 tables, one for each group:
1. **[in real life]** The first table uses the PSID men.
2. **[only in this module]** The second table uses the lottery men.

In [13]:
def moves(df, title):
    with_all = estimate(df, "re78", COLUMNS, "", quiet=True)[0]
    full = smf.ols("re78 ~ treat + " + " + ".join(COLUMNS), df).fit()
    rows = []
    for column in COLUMNS:
        others = [other for other in COLUMNS if other != column]
        without = round(smf.ols("re78 ~ treat + " + " + ".join(others), df).fit().params["treat"], 1)
        difference = smf.ols(f"{column} ~ treat + " + " + ".join(others), df).fit().params["treat"]
        rows.append({"Column left out": NAMES[column], "Estimate without": f"{without:,.1f}",
                     "Move = without - with all 8": f"{without - with_all:,.1f}", "Coefficient": f"{full.params[column]:,.6f}",
                     "Column difference": f"{difference:,.4f}", "Coefficient x Column difference": f"{full.params[column] * difference:,.1f}"})
    table = pd.DataFrame(rows)
    print(f"{title}, estimate with all 8 columns = {with_all:,.1f}. Each column left out, one at a time:")
    print()
    common.show(table)
    sizes = table["Move = without - with all 8"].str.replace(",", "").astype(float).abs()
    return sizes.max(), table["Column left out"][sizes.idxmax()], with_all

def share(result):
    largest, column, with_all = result
    print(f"Largest move, ignoring the sign = {largest:,.1f}, leaving out {column}")
    print()
    print("Largest move as a share of the estimate =")
    print("  = (largest move, ignoring the sign) / (estimate with all 8 columns) =")
    print(f"  = {largest:,.1f} / {with_all:,.1f} = {largest / with_all:.1%}")

psid_moves = moves(psid, "PSID men as the comparison group")

PSID men as the comparison group, estimate with all 8 columns = 751.9. Each column left out, one at a time:

Column left out        Estimate without  Move = without - with all 8   Coefficient  Column difference  Coefficient x Column difference
age                             1,089.1                        337.2    -83.565585            -4.0340                            337.1
years of education              1,134.2                        382.3    592.610194             0.6450                            382.2
black                             554.6                       -197.3   -570.927981             0.3457                           -197.4
hispanic                          870.2                        118.3  2,163.281159             0.0547                            118.3
married                           105.1                       -646.8  1,240.519511            -0.5215                           -646.9
no high-school degree             859.8                        107.9    590.46693

In [14]:
share(psid_moves)

Largest move, ignoring the sign = 1,012.1, leaving out 1975 earnings

Largest move as a share of the estimate =
  = (largest move, ignoring the sign) / (estimate with all 8 columns) =
  = 1,012.1 / 751.9 = 134.6%


In [15]:
lottery_moves = moves(experiment, "Lottery men as the comparison group")

Lottery men as the comparison group, estimate with all 8 columns = 1,676.3. Each column left out, one at a time:

Column left out        Estimate without  Move = without - with all 8    Coefficient  Column difference  Coefficient x Column difference
age                             1,688.8                         12.5      55.316676             0.2243                             12.4
years of education              1,623.5                        -52.8     395.734305            -0.1336                            -52.9
black                           1,711.4                         35.1  -2,159.522158            -0.0162                             35.0
hispanic                        1,671.0                         -5.3     164.032698            -0.0327                             -5.4
married                         1,673.5                         -2.8    -138.725286             0.0206                             -2.9
no high-school degree           1,682.6                          6.3  

In [16]:
share(lottery_moves)

Largest move, ignoring the sign = 52.8, leaving out years of education

Largest move as a share of the estimate =
  = (largest move, ignoring the sign) / (estimate with all 8 columns) =
  = 52.8 / 1,676.3 = 3.1%


**Observation:** with the PSID men, leaving out 1975 earnings moves the estimate by (−$1,012.1), 134.6% of the $751.9 estimate: one column in the data moves the estimate past 0, to (−$260.2). So a missing column that moved the estimate as far as 1975 earnings does, in either direction, would put the estimate anywhere from 751.9 − 1,012.1 = (−$260.2) to 751.9 + 1,012.1 = $1,764.0. With the lottery men, no column moves the estimate by more than $52.8, 3.1% of the $1,676.3 estimate: the lottery made the groups differ only by chance, so every column difference is small, and so is every product. Whether a missing column as strong as 1975 earnings exists, the data can't say: the check measures how far one column can move the estimate, not what a missing column did.

**Conclusion:** check 3 warns with the PSID men, a bad sign: one column in the data moves the PSID estimate past 0, so a missing column as strong could too.

## What do the three checks say, and were they right?

**[only in this module]** The table puts each check's result side by side, for the lottery men and for the PSID men.

In [17]:
def interval(result):
    return f"{result[0]:,.1f} ({result[1]:,.1f} to {result[2]:,.1f})"

def warns(result):
    return "no, 0 inside" if result[1] <= 0 <= result[2] else "yes, 0 outside"

print("The three checks, lottery men and PSID men, each as the comparison group:")
print()
common.show(pd.DataFrame({
    "Check": ["1. Placebo outcome, 1975 earnings", "2. Stand-in group", "3. Largest move / estimate"],
    "Lottery men": [interval(lottery_placebo), "can't run: lottery men vs lottery men", f"{lottery_moves[0]:,.1f} / {lottery_moves[2]:,.1f} = {lottery_moves[0] / lottery_moves[2]:.1%}"],
    "Warns?": [warns(lottery_placebo), "—", "no, below 100%"],
    "PSID men": [interval(psid_placebo), interval(psid_bias), f"{psid_moves[0]:,.1f} / {psid_moves[2]:,.1f} = {psid_moves[0] / psid_moves[2]:.1%}"],
    "Warns? ": [warns(psid_placebo), warns(psid_bias), "yes, above 100%"],
}))

The three checks, lottery men and PSID men, each as the comparison group:

Check                              Lottery men                            Warns?          PSID men                       Warns?
1. Placebo outcome, 1975 earnings  317.9 (-123.7 to 759.5)                no, 0 inside    -1,781.6 (-3,040.5 to -522.7)  yes, 0 outside
2. Stand-in group                  can't run: lottery men vs lottery men  —               -1,157.8 (-2,789.3 to 473.7)   no, 0 inside
3. Largest move / estimate         52.8 / 1,676.3 = 3.1%                  no, below 100%  1,012.1 / 751.9 = 134.6%       yes, above 100%


**[only in this module]** Now the right answer, hidden since the section *What estimate do the checks test?*: the PSID estimate's miss is (−$1,042.4). Check 2's bias estimate is an estimate of that miss, so the output compares the two.

In [18]:
print("PSID men, bias estimate - miss =")
print("  = (check 2's bias estimate) - (miss) =")
print(f"  = {psid_bias[0]:,.1f} - ({miss:,.1f}) = {psid_bias[0] - miss:,.1f} dollars per year")

PSID men, bias estimate - miss =
  = (check 2's bias estimate) - (miss) =
  = -1,157.8 - (-1,042.4) = -115.4 dollars per year


**Observation:** with the PSID men, two of the three checks warn. Check 1 finds an effect of (−$1,781.6) on 1975 earnings, which the program couldn't change. Check 3 finds that one column moves the estimate by 134.6% of the estimate. Check 2 doesn't warn, yet its bias estimate, (−$1,157.8), is $115.4 further below 0 than the miss, (−$1,042.4), about a tenth of the miss. With the lottery men, no check warns.

**[only in this module]** The checks were right to doubt the PSID regression estimate: the estimate is $1,042.4 below the right answer. The estimate's interval, (−1,042.1) to 2,545.9, holds the right answer ($1,794.3), but also 0, so the PSID estimate is also too uncertain to say whether the program raised earnings at all.

**Conclusion:** together the checks showed, without the right answer, that the PSID estimate can't be trusted, and the right answer shows they were right.

## Exercise — the CPS men as the comparison group

The PSID men are not the only survey sample in the file. The exercise repeats the regression estimate and the three checks with the 15,992 CPS men as the comparison group, then compares the checks with the right answer. The outputs give every number the questions need, in the order of the module's sections.

- **[in real life]** The exercise runs the regression estimate and the three checks, with the lottery men as check 2's stand-in group, as in the module.
- **[only in this module]** The exercise then compares the checks with the right answer.

Write your answers first, then read the answers below.

In [19]:
cps_estimate = estimate(cps, "re78", COLUMNS, "CPS men as the comparison group, regression estimate")

CPS men as the comparison group, regression estimate (16,177 men) =
  = coefficient of 'took the program' (one regression of 1978 earnings on 'took the program' and 8 columns) =
  = 699.1 dollars per year
The result, 699.1, is b in: a + (b x took the program) + (c1 x age) + ... + (c8 x 1975 earnings)

Standard error, returned by the fitting function = 547.6

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = 699.1 ± (1.96 x 547.6) = 699.1 ± 1,073.3 =
  = -374.2 to 1,772.4


In [20]:
cps_placebo = estimate(cps, "re75", SEVEN, "Check 1, CPS men as the comparison group, placebo estimate (effect on 1975 earnings)")

Check 1, CPS men as the comparison group, placebo estimate (effect on 1975 earnings) (16,177 men) =
  = coefficient of 'took the program' (one regression of 1975 earnings on 'took the program' and 7 columns) =
  = -1,384.2 dollars per year
The result, -1,384.2, is b in: a + (b x took the program) + (c1 x age) + ... + (c7 x 1974 earnings)

Standard error, returned by the fitting function = 353.9

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = -1,384.2 ± (1.96 x 353.9) = -1,384.2 ± 693.6 =
  = -2,077.8 to -690.6


In [21]:
second_cps = pd.concat([lottery.assign(treat=1), cps[cps.treat == 0]], ignore_index=True)
cps_bias = estimate(second_cps, "re78", COLUMNS, "Check 2, lottery men against CPS men, bias estimate", column="lottery man")

Check 2, lottery men against CPS men, bias estimate (16,252 men) =
  = coefficient of 'lottery man' (one regression of 1978 earnings on 'lottery man' and 8 columns) =
  = -1,047.2 dollars per year
The result, -1,047.2, is b in: a + (b x lottery man) + (c1 x age) + ... + (c8 x 1975 earnings)

Standard error, returned by the fitting function = 471.5

95% confidence interval =
  = (estimate) ± (1.96 x standard error) =
  = -1,047.2 ± (1.96 x 471.5) = -1,047.2 ± 924.1 =
  = -1,971.3 to -123.1


In [22]:
cps_moves = moves(cps, "Check 3: CPS men as the comparison group")

Check 3: CPS men as the comparison group, estimate with all 8 columns = 699.1. Each column left out, one at a time:

Column left out        Estimate without  Move = without - with all 8  Coefficient  Column difference  Coefficient x Column difference
age                               781.6                         82.5  -101.855073            -0.8099                             82.5
years of education                762.1                         63.0   160.186355             0.3929                             62.9
black                              97.0                       -602.1  -836.947293             0.7194                           -602.1
hispanic                          692.5                         -6.6  -218.318388             0.0305                             -6.7
married                           684.3                        -14.8    73.075761            -0.2031                            -14.8
no high-school degree             754.2                         55.1   372.2375

In [23]:
share(cps_moves)

Largest move, ignoring the sign = 651.6, leaving out 1975 earnings

Largest move as a share of the estimate =
  = (largest move, ignoring the sign) / (estimate with all 8 columns) =
  = 651.6 / 699.1 = 93.2%


### Questions

1. Does check 1, the placebo outcome, warn with the CPS men?
2. Does check 2, the stand-in group, warn with the CPS men? With the PSID men, check 2 didn't warn: what differs?
3. Is the CPS estimate fragile by the rule of check 3?
4. **[only in this module]** What is the CPS estimate's miss, and how far is check 2's bias estimate from that miss?

### Answers

**Answer 1.** Yes. The placebo estimate is (−$1,384.2), and the interval, (−2,077.8) to (−690.6), doesn't include 0: the treated men earned $1,384.2 less in 1975 than CPS men alike in the other 7 columns. A bad sign: the columns don't make the CPS men like the treated men, so the CPS regression estimate is in doubt.

**Answer 2.** Yes. The bias estimate is (−$1,047.2), and the interval, (−1,971.3) to (−123.1), doesn't include 0.

- With the PSID men, the bias estimate was (−$1,157.8), and the interval, (−2,789.3) to 473.7, held 0.
- Difference between the two bias estimates = (CPS men's bias estimate) − (PSID men's bias estimate) = (−1,047.2) − (−1,157.8) = $110.6, about a tenth of either bias estimate.
- What differs is the interval's width: the standard error, returned by the fitting function, is 471.5 with the CPS men against 832.4 with the PSID men, so the interval's half-width is 1.96 × 471.5 = 924.1 against 1.96 × 832.4 = 1,631.5. A standard error usually shrinks as the number of men in the regression grows: the CPS regression has 16,252 men against 2,750.
- So with the CPS men, check 2 warns that the CPS regression estimate has a bias, with a bias estimate below 0. Check 1 warns too.

**Answer 3.** No, but only just. The largest move, from leaving out 1975 earnings, is (−$651.6): 651.6 ÷ 699.1 = 93.2% of the estimate, below 100%.

**Answer 4.** **[only in this module]**

- Miss, CPS men = (regression estimate, CPS men) − (right answer) = 699.1 − 1,794.3 = (−$1,095.2)
- Bias estimate − miss = (−1,047.2) − (−1,095.2) = $48.0

**What the answers show:** with the CPS men, checks 1 and 2 warn, and check 3 doesn't. The next comparison is a different comparison from check 3: it uses the right answer to grade check 3's rule after the fact. The CPS miss, $1,095.2 in size, ignoring the sign, is larger than the largest move from any single column in the data, $651.6 (answers 3 and 4). Part of the miss can be chance, but the moves of the columns in the data understated the miss here, so check 3 not warning doesn't show that the estimate is close to the right answer.

## Interview angle

> "Matching says users who turned on feature X spend 8% more. An experiment isn't possible. How would you decide whether to trust the 8%?"

Matching pairs each user who turned on feature X with similar users who didn't. A strong answer: the right answer isn't known, so check the 8% in ways that need no right answer.

- A placebo outcome: run the same matching on spend in the month before the users turned on feature X. Feature X can't have changed that spend, so the estimate should be 0. An interval without 0 means the matched users differ from the users who turned on feature X in ways the matching's columns, such as past spend and sign-up date, miss.
- A stand-in group: if some users wanted feature X but didn't get it, for example because of a waiting list, compare those users with the matched users. Neither group used feature X, so the estimate is a bias estimate. A narrow interval matters: a wide interval can hold a large bias and still include 0.
- Sensitivity to a missing column: leave out each column the matching uses, one at a time, and see how far the 8% moves. If one column moves the 8% to 0, a column the data doesn't have, such as how much a user needs feature X, could do the same.
- No single check proves the 8% right. A check that warns is a reason to doubt the 8%, and a check that doesn't warn shows only that the check couldn't find a problem.

## Key takeaways

- Outside an experiment, the right answer is never known, so the checks must need none: a placebo outcome, a stand-in group, and sensitivity to a missing column.
- Here, the right answer was known from the lottery, $1,794.3, and the regression estimate with the PSID men, $751.9, missed by (−$1,042.4).
- A placebo outcome is an outcome the program couldn't change. With the PSID men, the regression found an effect of (−$1,781.6) on 1975 earnings, with an interval without 0, so the check warned. With the lottery men, the check didn't warn.
- A stand-in group compares two groups that both didn't take the program, so the estimate is a bias estimate. With the PSID men, the bias estimate was (−$1,157.8), $115.4 from the miss, but the interval held 0, so the check didn't warn. With the CPS men, a narrower interval made the same check warn.
- Move from leaving out a column = (the column's coefficient) × (the column difference between the groups, holding the other columns fixed). With the PSID men, leaving out 1975 earnings moved the estimate by (−$1,012.1), more than the whole estimate, so the estimate was fragile.
- A check that doesn't warn proves nothing: with the CPS men, check 3 didn't warn, yet the CPS estimate missed by (−$1,095.2). Several checks together, not any one, show whether to trust an estimate.

## Appendix: why the move is exactly (coefficient × column difference)

The section *Check 3 of 3: how far could a missing column move the estimate?* uses the formula move = (coefficient × column difference). This appendix shows why the formula holds exactly, written for 1975 earnings.

1. The 8-column regression:

   **formula**: 1978 earnings = a + (b × took the program) + (c × other 7 columns) + (c8 × 1975 earnings) + error

   (c × other 7 columns) is short for (c1 × age) + … + (c7 × 1974 earnings). The fitting function returns the one set of numbers a, b, c, c8 whose squared errors add up to the smallest total. Numbers picked that way, in any regression, always leave errors with mean 0 and no correlation with any column in the regression (here: took the program, the other 7 columns and 1975 earnings). The next lines use that property.
2. Check 1's regression:

   **formula**: 1975 earnings = check 1's a + (column difference × took the program) + (check 1's c × other 7 columns) + leftover

   The leftovers have no correlation with took the program or the other 7 columns, and mean 0.
   - Check 1's numbers differ from line 1's, so they carry "check 1's" in front.
   - The column difference is check 1's placebo estimate: the difference in 1975 earnings, holding the other 7 columns fixed.
3. Put line 2 into line 1 in place of 1975 earnings, in 3 steps:
   - Step 1: line 1's piece (c8 × 1975 earnings), with line 2's right side in place of 1975 earnings:
     - c8 × 1975 earnings =
     - = c8 × (check 1's a + (column difference × took the program) + (check 1's c × other 7 columns) + leftover) =
     - = (c8 × check 1's a) + ((c8 × column difference) × took the program) + ((c8 × check 1's c) × other 7 columns) + (c8 × leftover)
   - Step 2: add each of step 1's 4 pieces to line 1's piece of the same kind:
     - a  →  a + (c8 × check 1's a)
     - b × took the program  →  (b + (c8 × column difference)) × took the program
     - c × other 7 columns  →  (c + (c8 × check 1's c)) × other 7 columns
     - error  →  (c8 × leftover) + error
   - Step 3: line 1, with 1975 earnings replaced =
     - = (a + (c8 × check 1's a)) + ((b + (c8 × column difference)) × took the program) + ((c + (c8 × check 1's c)) × other 7 columns) + ((c8 × leftover) + error)
4. Step 3's last piece is (c8 × leftover) + error. It has mean 0 and no correlation with took the program or the other 7 columns, because each of its 2 parts has mean 0 and no correlation with took the program or the other 7 columns:
   - Line 2 states this for the leftover, and multiplying the leftover by c8 keeps it true.
   - Line 1 states this for the error.

   Adding two parts that each have mean 0 and no correlation with a column gives a sum with mean 0 and no correlation with that column.
5. Step 3 writes 1978 earnings with took the program and the other 7 columns only, and line 4 shows that step 3's errors have mean 0 and no correlation with those columns. Only the fitting function's numbers leave errors like that, so step 3's numbers are the ones the regression without 1975 earnings returns: the estimate without 1975 earnings is b + (c8 × column difference).
6. Move = (estimate without 1975 earnings) − (estimate with 1975 earnings) = (b + (c8 × column difference)) − b = c8 × column difference = (coefficient of 1975 earnings) × (column difference in 1975 earnings).